### In this notebook, we preprocess our HI-C data to keep only 25% of total edges. We load in the coordinates of different regions, bin the regions by distance based on the coordinates, subtract the 75th percentile distance of the bins from each regions HI-C value, and then discard all negative values.

In [1]:
import numpy as np
from scipy.spatial import distance_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#load in coordinates

ec_coords = np.genfromtxt('data/GBM39_5k_coordinates.txt')
hsr_coords = np.genfromtxt('data/GBM39HSR_5k_coordinates.txt')

In [3]:
ec_hic = np.load('data/GBM39ec_5k_collapsed_matrix.npy')
hsr_hic = np.load('data/GBM39HSR_5k_collapsed_matrix.npy')
ec_hic.shape, hsr_hic.shape

((251, 251), (251, 251))

In [4]:
#compute the distances between all pairs of regions

dist_matrix = distance_matrix(ec_coords, ec_coords)
dist_matrix

array([[0.        , 0.01543365, 0.03082205, ..., 0.05539887, 0.04083208,
        0.02575682],
       [0.01543365, 0.        , 0.01546108, ..., 0.07068806, 0.0562215 ,
        0.04113406],
       [0.03082205, 0.01546108, 0.        , ..., 0.08573747, 0.07148551,
        0.05635993],
       ...,
       [0.05539887, 0.07068806, 0.08573747, ..., 0.        , 0.01536143,
        0.02971846],
       [0.04083208, 0.0562215 , 0.07148551, ..., 0.01536143, 0.        ,
        0.015355  ],
       [0.02575682, 0.04113406, 0.05635993, ..., 0.02971846, 0.015355  ,
        0.        ]])

In [5]:
def build_adj(coords, hic, bin_width = 0.05):
    dist_matrix = distance_matrix(coords, coords) #compute distances between all pairs of regions
    dist_matrix_rounded = np.ceil(dist_matrix / bin_width) * bin_width #bin the distances by rounding to the nearest bin_width
    unique_distances = np.unique(dist_matrix_rounded) 
    
    distance_medians = {}
    norm_hic = hic.copy()

    for d in unique_distances:
        #find all binned values that equal the current distance and find the 75th percentile of these values
        mask = dist_matrix_rounded == d
        distance_medians[d] = np.percentile(hic[mask], 75)

        #subtract the 75th percentile from each value and remove negative values
        norm_hic[mask] = np.clip(norm_hic[dist_matrix_rounded == d] - distance_medians[d], a_min=0, a_max=None)

    return norm_hic

In [6]:
ec_adj = build_adj(ec_coords, ec_hic)
hsr_adj = build_adj(hsr_coords, hsr_hic)

In [7]:
print(f'Proportion of edges removed (ec): {np.mean(ec_adj == 0)}')
print(f'Proportion of edges removed (hsr): {np.mean(hsr_adj == 0)}')

Proportion of edges removed (ec): 0.7499722226631323
Proportion of edges removed (hsr): 0.7499880954270567


In [8]:
np.save('data/hsr_adj_mat_t25.npy', hsr_adj)
np.save('data/ec_adj_mat_t25.npy', ec_adj)